In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
import sys

project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))

PROCESSED_DATA_DIR = project_root / "data" / "processed"
FIGURES_DIR        = project_root / "figures"
PCA_DIR            = project_root / "data" / "pca"
PCA_DIR.mkdir(parents=True, exist_ok=True)

# Must match notebook 02 exactly
LOG_MONEYNESS_GRID = np.linspace(np.log(0.80), np.log(1.20), 50)
TTM_GRID           = np.linspace(14/365, 1.0, 50)
GRID_SHAPE         = (50, 50)
VECTOR_SIZE        = 50 * 50  # 2500

print("Config loaded.")
print(f"Processed data dir: {PROCESSED_DATA_DIR}")
print(f"PCA output dir:     {PCA_DIR}")

Config loaded.
Processed data dir: /Users/kennethdavis/Desktop/Personal/Personal Projects/IV Surface/Implied-Volatility-Surface-PCA/data/processed
PCA output dir:     /Users/kennethdavis/Desktop/Personal/Personal Projects/IV Surface/Implied-Volatility-Surface-PCA/data/pca


In [2]:
def nadaraya_watson(m_grid, t_grid, m_data, t_data, iv_data, h_m=0.05, h_t=0.1):
    surface = np.zeros((len(t_grid), len(m_grid)))
    for i, t in enumerate(t_grid):
        for j, m in enumerate(m_grid):
            w = (np.exp(-0.5 * ((m_data - m) / h_m)**2) *
                 np.exp(-0.5 * ((t_data - t) / h_t)**2))
            if w.sum() > 0:
                surface[i, j] = np.dot(w, iv_data) / w.sum()
            else:
                surface[i, j] = np.nan
    return surface

In [3]:
processed_files = sorted(PROCESSED_DATA_DIR.glob("SPY_IV_*.csv"))
print(f"Found {len(processed_files)} processed files\n")

surface_matrix = []
dates = []
skipped = []

for f in processed_files:
    date = f.stem.replace("SPY_IV_", "")
    try:
        df = pd.read_csv(f)

        # Rebuild surface from processed IV data
        surface = nadaraya_watson(
            LOG_MONEYNESS_GRID, TTM_GRID,
            df["log_moneyness"].values,
            df["TTM"].values,
            df["IV"].values
        )

        # Skip if surface has too many NaNs
        nan_pct = np.isnan(surface).mean()
        if nan_pct > 0.1:
            print(f"[WARN] {date} — {nan_pct:.1%} NaN in surface, skipping")
            skipped.append(date)
            continue

        # Fill any remaining NaNs with surface mean
        surface = np.where(np.isnan(surface), np.nanmean(surface), surface)

        # Flatten to vector and append
        surface_matrix.append(surface.flatten())
        dates.append(date)

    except Exception as e:
        print(f"[ERR] {date} — {e}")
        skipped.append(date)

surface_matrix = np.array(surface_matrix)
print(f"\nSurface matrix shape: {surface_matrix.shape}")
print(f"Days included: {len(dates)}")
print(f"Days skipped:  {len(skipped)}")

Found 199 processed files


Surface matrix shape: (199, 2500)
Days included: 199
Days skipped:  0


In [4]:
# Demean the matrix (PCA standard practice)
mean_surface_vec = surface_matrix.mean(axis=0)
X = surface_matrix - mean_surface_vec

# Fit PCA
pca = PCA()
scores = pca.fit_transform(X)

explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("Top 10 principal components:")
print(f"{'PC':<6} {'Explained Var':>15} {'Cumulative':>12}")
print("-" * 35)
for i in range(10):
    print(f"PC{i+1:<4} {explained_variance[i]:>14.1%} {cumulative_variance[i]:>11.1%}")

Top 10 principal components:
PC       Explained Var   Cumulative
-----------------------------------
PC1             95.2%       95.2%
PC2              3.0%       98.1%
PC3              1.0%       99.1%
PC4              0.5%       99.6%
PC5              0.2%       99.8%
PC6              0.1%       99.9%
PC7              0.0%       99.9%
PC8              0.0%       99.9%
PC9              0.0%      100.0%
PC10             0.0%      100.0%


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual explained variance
axes[0].bar(range(1, 16), explained_variance[:15] * 100, color='steelblue', alpha=0.8)
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance (%)")
axes[0].set_title("Scree Plot")
axes[0].set_xticks(range(1, 16))

# Cumulative explained variance
axes[1].plot(range(1, 16), cumulative_variance[:15] * 100, 
             marker='o', color='steelblue', linewidth=2)
axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90%')
axes[1].axhline(y=95, color='orange', linestyle='--', alpha=0.5, label='95%')
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance (%)")
axes[1].set_title("Cumulative Explained Variance")
axes[1].set_xticks(range(1, 16))
axes[1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / "scree_plot.png", dpi=150)
plt.close(fig)
print("Scree plot saved.")

Scree plot saved.


In [6]:
M, T = np.meshgrid(LOG_MONEYNESS_GRID, TTM_GRID)
n_components = 3

fig = plt.figure(figsize=(18, 5))
for i in range(n_components):
    ax = fig.add_subplot(1, n_components, i+1, projection='3d')
    loading_surface = pca.components_[i].reshape(GRID_SHAPE)
    ax.plot_surface(M, T, loading_surface, cmap='RdBu', alpha=0.8)
    ax.set_xlabel("Log-Moneyness")
    ax.set_ylabel("TTM (years)")
    ax.set_zlabel("Loading")
    ax.set_title(f"PC{i+1} ({explained_variance[i]:.1%} var)")

plt.suptitle("Principal Component Loadings — SPY IV Surface 2025", y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "pc_loadings.png", dpi=150, bbox_inches='tight')
plt.close(fig)
print("PC loadings plot saved.")

PC loadings plot saved.


In [7]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
dates_dt = pd.to_datetime(dates)

for i in range(3):
    axes[i].plot(dates_dt, scores[:, i], linewidth=1.2, color='steelblue')
    axes[i].axhline(y=0, color='black', linewidth=0.5, alpha=0.5)
    axes[i].set_ylabel(f"PC{i+1} Score")
    axes[i].set_title(f"PC{i+1} Score Time Series ({explained_variance[i]:.1%} variance explained)")

axes[-1].set_xlabel("Date")
plt.suptitle("Principal Component Scores — SPY IV Surface 2025")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "pc_scores_timeseries.png", dpi=150)
plt.close(fig)
print("PC scores time series saved.")

PC scores time series saved.


In [8]:
from matplotlib.animation import FuncAnimation

M, T = np.meshgrid(LOG_MONEYNESS_GRID, TTM_GRID)
mean_surface = mean_surface_vec.reshape(GRID_SHAPE)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

def update(frame):
    ax.cla()
    reconstructed = (mean_surface + 
                     (surface_matrix[frame] - mean_surface_vec).reshape(GRID_SHAPE))
    ax.plot_surface(M, T, reconstructed, cmap='viridis', alpha=0.8)
    ax.set_xlabel("Log-Moneyness")
    ax.set_ylabel("TTM (years)")
    ax.set_zlabel("Implied Volatility")
    ax.set_title(f"SPY IV Surface — {dates[frame]}")
    ax.set_zlim(
        np.nanmin(surface_matrix) * 0.9,
        np.nanmax(surface_matrix) * 1.1
    )

anim = FuncAnimation(fig, update, frames=len(dates), interval=100)
anim.save(FIGURES_DIR / "iv_surface_animation.gif", 
          writer='pillow', fps=10, dpi=100)
plt.close(fig)
print("Animation saved.")

Animation saved.


In [9]:
# Save PC scores
scores_df = pd.DataFrame(
    scores[:, :10],
    index=dates,
    columns=[f"PC{i+1}" for i in range(10)]
)
scores_df.index.name = "date"
scores_df.to_csv(PCA_DIR / "pc_scores.csv")
print(f"PC scores saved: {scores_df.shape}")

# Save explained variance
var_df = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(explained_variance))],
    "explained_variance": explained_variance,
    "cumulative_variance": cumulative_variance
})
var_df.to_csv(PCA_DIR / "explained_variance.csv", index=False)
print(f"Explained variance saved.")

# Save mean surface
mean_df = pd.DataFrame(
    mean_surface_vec.reshape(GRID_SHAPE),
    index=np.round(TTM_GRID, 4),
    columns=np.round(LOG_MONEYNESS_GRID, 4)
)
mean_df.to_csv(PCA_DIR / "mean_surface.csv")
print(f"Mean surface saved.")

print("\nAll PCA results saved to data/pca/")

PC scores saved: (199, 10)
Explained variance saved.
Mean surface saved.

All PCA results saved to data/pca/
